# LlamaIndex 实现 RAG —— 完整可运行教程（DeepSeek + 本地 Embedding）

## 这个教程是什么？

从零开始，用 LlamaIndex 搭建一个完整的 RAG（检索增强生成）系统。每一行代码都能运行，每个概念都有解释。

## 你会学到什么？

| 步骤 | 内容 |
|------|------|
| 环境准备 | 安装所有依赖 |
| 配置模型 | DeepSeek LLM + 本地 HuggingFace Embedding |
| 数据连接器 | 加载 PDF 文档（PyMuPDF） |
| 文档切分 | SentenceSplitter 的分块策略 |
| 向量索引 | FAISS 向量数据库 |
| 基础问答 | RAG 最核心的检索+生成流程 |
| Debug 检索 | 单独查看检索质量 |
| 多轮对话 | ChatMemoryBuffer 实现上下文记忆 |
| Query Engines | Routing / Rewriting / Planning 概念 |
| Agent 工具 | LLM 自主选择工具回答问题 |

## 你需要准备什么？

- **DeepSeek API Key**：[platform.deepseek.com](https://platform.deepseek.com) 注册获取
- **data/ 目录下的 PDF 文档**：项目已自带（同仁堂研报、世运电路中报点评、大模型产业报告）
- **Python 环境**：建议用 conda 创建独立环境

## 怎么运行？

**从上到下，一个 Cell 一个 Cell 执行。** 每个代码 Cell 之前都有解释，告诉你这一步在做什么、为什么这么做。

---

## 先认识一下 LlamaIndex

LlamaIndex 是一个专门为 RAG 设计的 LLM 数据框架。它的核心思路是：**把外部数据变成 LLM 能理解的东西**。

广泛可用的大模型（如 GPT、DeepSeek）通常只在公开数据上预训练过，但实际应用往往需要私有或特定领域的数据——这些数据散布在 PDF、数据库、API、网页等各处。LlamaIndex 负责把这些数据提取、结构化、索引，然后高效地喂给 LLM。

LlamaIndex 的五大模块：

```
数据连接器           数据索引             查询引擎           数据代理           应用集成
Data Connectors → Data Indexes → Query Engines → Data Agents → App Integrations
     ↓                 ↓                 ↓               ↓               ↓
从各种数据源      对文档分chunk        执行语义搜索    LLM 自主选择工具     对接外部系统
提取数据          向量化存储           处理查询       执行复杂任务        （LLM、BI 等）
```

| 模块 | 做什么 | 本教程对应 |
|------|--------|-----------|
| **Data Connectors** | 从 PDF / 数据库 / API / 网页等加载数据 | 第三步：加载文档 |
| **Data Indexes** | 对文档分 chunk、向量化、建索引 | 第四、五步：切分 + 向量索引 |
| **Query Engines** | 检索 + 生成答案，支持 Routing / Rewriting / Planning | 第六、七步：问答 + Debug |
| **Data Agents** | LLM 自主选择工具，多步骤推理 | 第十步：Agent |
| **App Integrations** | 对接外部 LLM、数据库等 | 第二步：配置 DeepSeek |

下面我们逐个实现。

---

## 第一步：安装依赖（只需执行一次）

以下是本教程需要的包。**注意**：如果你在国内，建议先设置 HuggingFace 镜像，否则后续下载 Embedding 模型可能失败。

```python
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"  # 国内镜像
```

> 如果已经装过这些包，运行这个 Cell 会很快跳过，不用担心重复安装。

| 包名 | 用途 |
|------|------|
| `llama-index-core` | LlamaIndex 核心框架 |
| `llama-index-llms-openai-like` | 连接 DeepSeek（OpenAI 兼容接口） |
| `llama-index-embeddings-huggingface` | 本地运行 Embedding 模型 |
| `llama-index-readers-file` | 文件读取器集合 |
| `llama-index-vector-stores-faiss` | FAISS 向量数据库适配 |
| `faiss-cpu` | FAISS 向量库 CPU 版本 |
| `pymupdf` | 高质量 PDF 解析 |
| `nest-asyncio` | Jupyter 异步支持 |
| `huggingface-hub` | HuggingFace 模型下载和管理 |

In [ ]:
# 核心框架
!pip install -q llama-index-core

# DeepSeek 接口 + 本地 Embedding
!pip install -q llama-index-llms-openai-like llama-index-embeddings-huggingface

# 文档读取
!pip install -q llama-index-readers-file pymupdf

# 向量数据库
!pip install -q llama-index-vector-stores-faiss faiss-cpu

# Jupyter 异步 + HuggingFace 工具
!pip install -q nest-asyncio huggingface-hub

print(">>> 全部安装完成 <<<")

---

## 第二步：配置模型

我们需要两个模型：

| 模型类型 | 用什么 | 为什么 |
|---------|--------|--------|
| **LLM**（生成答案） | DeepSeek (`deepseek-chat`) | 便宜、中文好、OpenAI 兼容 |
| **Embedding**（文字→向量） | BAAI/bge-small-zh-v1.5（本地运行） | 免费、中文优化、512 维 |

### 为什么用本地 Embedding 而不是 API？

- DeepSeek 目前没有公开的 Embedding API
- 本地模型**完全免费**，不消耗 API 额度
- `bge-small-zh-v1.5` 专为中文优化，轻量快速
- 第一次运行会自动下载（约 100MB），之后缓存复用

### 国内用户特别注意

设置 `HF_ENDPOINT` 环境变量指向 HuggingFace 镜像，否则模型下载会超时。

### Windows 用户特别注意

如果运行时出现 `does not appear to have a file named pytorch_model.bin` 错误，说明 Windows 软链接问题导致缓存路径解析失败。解决办法：用 `huggingface_hub` 先下载模型到本地，然后传完整路径。如果模型已下载好了（本项目已自带），也可以跳过。

> **请把下面代码中的 API Key 换成你自己的！**

In [ ]:
import os
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# ============================================================
# !! 换成你自己的 DeepSeek API Key
# ============================================================
os.environ['DEEPSEEK_API_KEY'] = 'sk-your-deepseek-key-here'

# 国内用户：设置 HuggingFace 镜像，加速模型下载
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# ============================================================

# ---- LLM：DeepSeek ----
Settings.llm = OpenAILike(
    model="deepseek-chat",                        # DeepSeek V3 对话模型
    api_key=os.environ['DEEPSEEK_API_KEY'],
    api_base="https://api.deepseek.com/v1",       # DeepSeek API 地址，注意要有 /v1
    temperature=0.1,                               # 温度越低回答越稳定
    is_chat_model=True,                            # 必须指定！否则默认走 /completions 会报错
)

# ---- Embedding：本地模型 ----
# 方式一：直接传 HuggingFace repo ID（需要网络或已缓存）
# Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-zh-v1.5")

# 方式二：传本地完整路径（Windows 软链接问题时的解决方案）
from huggingface_hub import try_to_load_from_cache
cached = try_to_load_from_cache("BAAI/bge-small-zh-v1.5", "config.json")
if cached:
    import pathlib
    model_path = str(pathlib.Path(cached).parent)
    Settings.embed_model = HuggingFaceEmbedding(model_name=model_path)
else:
    Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-zh-v1.5")

print(">>> LLM + Embedding 配置完毕 <<<")
print(f"   LLM: DeepSeek (deepseek-chat)")
print(f"   Embedding: BAAI/bge-small-zh-v1.5 (本地运行)")
print(f"   向量维度: 512")

---

## 第三步：加载文档 —— Data Connectors

LlamaIndex 的 **Data Connectors** 模块负责从各种数据源拉取数据。内置的 `SimpleDirectoryReader` 可以读目录下的所有文件，通过 `file_extractor` 参数可以指定不同文件类型用不同的解析器。

### 这里用什么解析器？

- **PyMuPDFReader**：比默认 PDF 解析器好，能提取文本坐标信息，对中文 PDF 兼容性好
- 原教程还介绍了 **BeautifulSoupWebReader**（网页读取）和 **LlamaParse**（LlamaCloud 付费解析服务），可根据需要选用

### 代码中使用 `os.path.join(os.path.dirname(__file__), "data")` 的好处

这样不管终端当前目录在哪里，都能正确定位到脚本旁边的 `data/` 文件夹，避免 `Directory ./data does not exist` 错误。

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PyMuPDFReader

# 用 os.path 获取脚本所在目录，避免相对路径问题
data_dir = os.path.join(os.path.dirname(__file__), "data")

loader = SimpleDirectoryReader(
    input_dir=data_dir,
    file_extractor={".pdf": PyMuPDFReader()},
)
documents = loader.load_data()

print(f">>> 成功加载 {len(documents)} 个文档片段 <<<")
for doc in documents[:5]:  # 只打印前 5 个，太多会刷屏
    fname = doc.metadata.get('file_name', 'unknown')
    print(f"   - {fname}  ({len(doc.text):,} 字符)")
if len(documents) > 5:
    print(f"   ... 还有 {len(documents) - 5} 个")
print()
print(f"注：PyMuPDFReader 默认按页读取，所以文档数是页数而非文件数")

---

## 第四步：文档切分（Chunking）

### 为什么要切分？

- PDF 可能有几十页，LLM 一次读不完（上下文窗口有限）
- 即使能读完，内容太杂会让 LLM 找不到重点
- 切成小块后，检索时能**精准定位**到相关段落

### SentenceSplitter 的核心参数

| 参数 | 含义 | 这里设的值 | 原版教程的值 |
|------|------|-----------|------------|
| `chunk_size` | 每个 chunk 最大字符数 | 512 | 1024 |
| `chunk_overlap` | 相邻 chunk 重叠字符数 | 50 | 100 |
| `paragraph_separator` | 段落分隔符 | 默认 `\n\n` | `\n\n` |
| `secondary_chunking_regex` | 二次切分正则（如识别表格） | 默认不使用 | `<table>(.+?)</table>` |

### chunk_size 怎么选？

- **太大**：检索精度下降，一次喂给 LLM 的内容太多
- **太小**：信息碎片化，LLM 没有足够上下文
- **512 是实践中的常用起点**，中英文都适用

### chunk_overlap 为什么重要？

假如关键信息刚好从中间被切断：`"2023年营业收入为48.81亿 | 美元，同比增长..."`，如果没有 overlap，检索命中前半段，LLM 就看不到"美元"和"同比增长"。有 overlap 后，相邻 chunk 包含重叠部分，信息不会丢失。

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=512,        # 每个 chunk 最大 512 字符
    chunk_overlap=50,      # chunk 之间重叠 50 字符
    # paragraph_separator="\n\n",          # 可选：自定义段落分隔符
    # secondary_chunking_regex="<table>(.+?)</table>",  # 可选：HTML 表格识别
)

nodes = splitter.get_nodes_from_documents(documents)

print(f">>> 切分完成：{len(documents)} 个文档片段 -> {len(nodes)} 个 chunk <<<")

# 预览前两个 chunk
print()
print("--- 前两个 chunk 预览 ---")
for i, node in enumerate(nodes[:2]):
    if len(node.text) > 0:
        print(f"\n[Chunk {i+1}] ({len(node.text)} 字符):")
        print(node.text[:200])
        print("...")
    else:
        print(f"\n[Chunk {i+1}] (空 chunk，跳过)")

---

## 第五步：构建向量索引（FAISS）

### 这一步在做什么？

```
chunk 文字  →  Embedding 模型  →  向量（一串数字） →  存入 FAISS
                                                                ↓
用户提问    →  Embedding 模型  →  问题向量        →  在 FAISS 中找最相似的向量
```

### 什么是向量相似度？

- 「今天天气真好」和「今天天气真不错」→ 向量距离**近**（语义相似）
- 「今天天气真好」和「公司营收多少」→ 向量距离**远**（语义无关）

FAISS（Facebook AI Similarity Search）就是高效做这个"找最近邻居"工作的。

### 向量维度

`bge-small-zh-v1.5` 输出 **512 维**向量。这个数字必须和模型匹配：

| 模型 | 向量维度 |
|------|---------|
| `bge-small-zh-v1.5` | **512** |
| `bge-base-zh-v1.5` | 768 |
| `bge-large-zh-v1.5` | 1024 |
| OpenAI `text-embedding-3-small` | 1536 |

如果维度写错了，FAISS 会直接报错。

### 为什么要用外部向量库（FAISS）而不是 LlamaIndex 内置的？

LlamaIndex 默认用内存存储向量，数据量大时检索效率低。FAISS 是 Facebook 专门为向量检索优化的库，速度快、支持大规模数据。

In [ ]:
import faiss
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.faiss import FaissVectorStore

# 1. 创建 FAISS 索引
#    512 = bge-small-zh-v1.5 的向量维度
#    IndexFlatL2 = 使用 L2 距离（欧几里得距离）
d = 512
faiss_index = faiss.IndexFlatL2(d)
vector_store = FaissVectorStore(faiss_index=faiss_index)

# 2. 把 nodes（chunk）向量化，存入 FAISS
storage_context = StorageContext.from_defaults(vector_store=vector_store)
vector_index = VectorStoreIndex(nodes, storage_context=storage_context)

print(f">>> 向量索引构建完成 <<<")
print(f"   向量数量: {len(nodes)}")
print(f"   向量维度: {d}")
print(f"   距离算法: L2（欧几里得距离）")

---

## 第六步：基础问答 —— RAG 核心流程

这是 RAG 最核心的用法，四步走：

```
1. 用户提问
       ↓
2. 问题 → 向量 → FAISS 检索最相关的 chunk
       ↓
3. 检索到的 chunk + 用户问题 → 一起发给 LLM
       ↓
4. LLM 基于材料生成答案
```

`similarity_top_k=3` 表示每次检索最相关的 **3 个** chunk 喂给 LLM。这个值可以调：
- 太小（如 1）：信息不全，可能漏掉关键内容
- 太大（如 10）：信息冗余，可能超出 LLM 上下文窗口

In [ ]:
# 创建查询引擎
query_engine = vector_index.as_query_engine(similarity_top_k=3)

# 提问
question = "世运电路2023年上半年实现营业收入多少？"
response = query_engine.query(question)

print(f"Q: {question}")
print(f"A: {response}")
print()
print(f"(回答基于 {len(response.source_nodes)} 个检索到的 chunk)")

---

## 第七步：Debug —— 查看检索结果

如果 LLM 回答得不好，问题通常出在**检索阶段**，而不是生成阶段。

**检索质量 > 生成质量**：如果检索回来的 chunk 跟问题无关，LLM 再强也只能胡编。

这个 Cell 把"检索"和"生成"拆开，单独看检索结果。检查要点：
- chunk 内容跟问题相关吗？
- 相似度得分合理吗？（越小越相似，0.3~0.5 通常 OK，>1.0 基本不相关）
- 有没有更好的 chunk 没有被检索到？

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever

# 创建纯检索器（只检索，不生成答案）
retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=3)

question = "同仁堂安宫牛黄丸的市场价格"
retrieved_nodes = retriever.retrieve(question)

print(f"问题: {question}")
print(f"检索到 {len(retrieved_nodes)} 个 chunk：")

for i, node in enumerate(retrieved_nodes):
    print(f"\n{'='*60}")
    print(f"Chunk {i+1}  |  相似度得分: {node.score:.4f}（越小越相似）")
    print(f"{'='*60}")
    print(node.node.text[:400])
    print("...")

---

## 第八步：带记忆的多轮对话

之前的 `query_engine.query()` 每次都"失忆"——它不记得你上一轮问过什么。

加上 `ChatMemoryBuffer` 后，系统保留对话历史。这样就可以追问：
- 「世运电路营收多少？」→ 系统回答
- 「它同比增长了多少？」→ 系统知道「它」指世运电路

### 关键参数

- `token_limit=5000`：最多保留 5000 个 token 的历史，旧的会自动丢弃
- `system_prompt`：系统提示词，定义助手的行为边界
- `chat_mode="context"`：每次把检索结果 + 历史对话一起给 LLM

### 和 LangChain 的区别

在 LangChain 中，你需要手动维护 `chat_history` 列表传给 chain。LlamaIndex 的 `ChatMemoryBuffer` 自动处理这些，代码更简洁。

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

# 创建带记忆的聊天引擎
memory = ChatMemoryBuffer.from_defaults(token_limit=5000)

chat_engine = vector_index.as_chat_engine(
    chat_mode="context",
    memory=memory,
    system_prompt="你是一个基于检索结果回答问题的助手。如果检索不到相关信息，就诚实地说不知道。",
)

# ---- 第一轮 ----
q1 = "世运电路2023年上半年营业收入多少？"
response1 = chat_engine.chat(q1)
print(f">> 用户: {q1}")
print(f">> 助手: {response1}")
print()

# ---- 第二轮：用代词追问，测试系统是否还记得上下文 ----
q2 = "它同比增长了多少？"
response2 = chat_engine.chat(q2)
print(f">> 用户: {q2}")
print(f">> 助手: {response2}")

---

## 第九步（概念）：Query Engines 的高级技巧

这一步只有概念，没有代码。LlamaIndex 的 Query Engines 模块除了基础检索+生成，还支持三种高级技巧：

### 1. Query Routing（查询路由）

根据问题内容，**自动引导到最相关的知识库**去检索。

```
用户问题 → Router → 知识库A（产品文档）？
                  → 知识库B（财务报表）？
                  → 知识库C（技术支持）？
```

适用场景：你有多个不同领域的文档集合。

### 2. Query Rewriting（查询改写）

对用户问题进行**多角度重写**，提高检索命中率。

```
原始问题："这玩意多少钱？"
改写1: "这个产品的价格是多少？"
改写2: "该商品的售价是多少元？"
```

适用场景：用户提问比较口语化，或者问题太短。

### 3. Query Planning（查询规划）

把复杂问题**拆解成子问题**，逐步检索，最后汇总。

```
复杂问题："对比 A 公司和 B 公司 2023 年的营收和利润"
  → 子问题1: A 公司 2023 年营收
  → 子问题2: A 公司 2023 年利润
  → 子问题3: B 公司 2023 年营收
  → 子问题4: B 公司 2023 年利润
  → 汇总对比
```

适用场景：问题涉及多方面信息，单次检索不够。

> 这些技巧在原版 LlamaIndex 教程 PDF 中有详细介绍，本教程不展开实现，知道概念即可。

---

## 第十步：Agent —— 让 LLM 自己选工具

### 前面的问题

基础问答、Debug 检索、多轮对话，本质都是**同一种方式**：向量检索 → 喂给 LLM。但不同类型的问题，可能需要不同的处理方式。

### Agent 是什么？

你给 LLM 一个「工具箱」，LLM 看到问题后**自己判断该用哪个工具**。

### 这里的两个工具

| 工具 | 适用场景 | 实现方式 |
|------|---------|---------|
| **precise_search** | 具体事实问题："XX 营收多少？" | 向量检索 |
| **summary** | 整体性问题："总结一下 XX 公司" | 摘要索引（tree_summarize） |

### ReActAgent 的工作流程

```
用户问题 → Thought（思考该用什么工具）
         → Action（选择并调用工具）
         → Observation（观察工具返回结果）
         → Thought（还需要更多信息吗？）
         → ...（循环直到能回答）
         → Final Answer（最终回答）
```

`verbose=True` 会把上述整个过程打印出来，非常有助于理解 Agent 的决策逻辑。

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.core.tools import FunctionTool, QueryEngineTool
from llama_index.core import SummaryIndex

# ---- 工具1：精确检索 ----
def vector_query(query: str) -> str:
    engine = vector_index.as_query_engine(similarity_top_k=3)
    return engine.query(query)

vector_tool = FunctionTool.from_defaults(
    name="precise_search",
    fn=vector_query,
    description="对文档进行向量检索，适合回答具体事实性问题，例如「XX营收多少」「XX价格多少」"
)

# ---- 工具2：全文总结 ----
summary_index = SummaryIndex(nodes)
summary_engine = summary_index.as_query_engine(response_mode="tree_summarize")

summary_tool = QueryEngineTool.from_defaults(
    name="summary",
    query_engine=summary_engine,
    description="对全文进行概括总结，适合回答整体性问题，例如「总结一下XX」「这篇文章讲了什么」"
)

print(">>> 工具箱准备完毕 <<<")
print("   1. precise_search - 精确检索（查具体问题）")
print("   2. summary - 全文总结（看整体）")

---

## 第十一步：运行 Agent

### 注意事项

- **llama-index 0.14+ 的 Agent API 已改为异步**：旧版的 `agent.chat()` 不再可用，需用 `agent.run()` + `await`
- `verbose=True` 会打印完整思考过程（Thought → Action → Observation）
- Agent 每次运行可能选择不同的工具路径，这是正常的

In [ ]:
from llama_index.core.agent import ReActAgent
import asyncio

# 创建 Agent（新版 API：直接构造，不再用 .from_tools()）
agent = ReActAgent(
    tools=[vector_tool, summary_tool],
    llm=Settings.llm,
    verbose=True
)

async def run_tests():
    # ---- 测试1：整体性问题 → 预期用 summary 工具 ----
    print("=" * 60)
    print("测试1：整体性问题（预期用 summary）")
    print("=" * 60)
    handler = agent.run("总结一下兴证电子")
    response = await handler
    print()
    print(">>> 最终回答 <<<")
    print(response)
    print()

    # ---- 测试2：具体问题 → 预期用 precise_search 工具 ----
    print("=" * 60)
    print("测试2：具体问题（预期用 precise_search）")
    print("=" * 60)
    handler = agent.run("世运电路2023年同比增长多少")
    response = await handler
    print()
    print(">>> 最终回答 <<<")
    print(response)

asyncio.run(run_tests())

---

## 总结

恭喜！你完成了一个完整的 RAG 系统。回顾一下：

| 步骤 | 做了什么 | 对应的 LlamaIndex 模块 |
|------|---------|----------------------|
| 加载文档 | SimpleDirectoryReader + PyMuPDFReader | **Data Connectors** |
| 文档切分 | SentenceSplitter（chunk_size + overlap） | **Data Indexes** |
| 向量索引 | Embedding 模型 + FAISS 向量库 | **Data Indexes** |
| 基础问答 | `as_query_engine()` 检索 + 生成 | **Query Engines** |
| Debug | 单独查看检索结果 | **Query Engines** |
| 多轮对话 | ChatMemoryBuffer 上下文记忆 | **Query Engines** |
| Agent | ReActAgent 自主选择工具 | **Data Agents** |

### 本教程和原版的主要区别

| 对比项 | 原版教程 | 本教程 |
|--------|---------|--------|
| LLM | OpenAI (gpt-4o) | **DeepSeek** (deepseek-chat) |
| Embedding | OpenAI (ada-002, 付费) | **本地 BGE** (免费) |
| HuggingFace 镜像 | 无 | **HF_ENDPOINT** 国内加速 |
| Windows 兼容 | 未处理 | **软链接问题解决方案** |
| 代码结构 | 多个独立 demo，不能顺序跑 | **线性依赖，从头跑到尾** |
| Agent API | 旧版 `.from_tools().chat()` | **新版 构造 + async run()** |
| 运行验证 | 未验证 | **实际跑通** |

### 接下来可以尝试

- 换自己的 PDF 文档试试
- 调整 `chunk_size` 和 `similarity_top_k` 看效果变化
- 给 Agent 添加第三个工具（如计算器、联网搜索）
- 把 FAISS 换成 ChromaDB 或 Milvus
- 实现 Query Rewriting（查询改写）提高检索质量
- 实现 Query Routing（查询路由）处理多知识库场景